# ⚡ Vector Search at Scale: Zero to Hero — A Guided Lab

The Embeddings & Search lab taught **brute-force** cosine similarity — checking every document,
one by one. That's correct but doesn't scale: at 10 million vectors, brute-force search becomes
too slow for real-time use. This lab teaches the **approximate nearest neighbor (ANN)**
algorithms real vector databases (FAISS, Pinecone, Chroma, Weaviate) actually run internally.

**Beginner-first.** Every chapter explains the *concept* before any code. Prerequisite: the
Embeddings & Search lab (cosine similarity, vector basics).

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Worked example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. Why brute-force doesn't scale
2. The accuracy/speed tradeoff: approximate search
3. Clustering-based search: IVF (Inverted File Index)
4. Locality-Sensitive Hashing (LSH)
5. Graph-based search: HNSW (the industry standard)
6. Product Quantization (compressing vectors)
7. Combining techniques: IVF + PQ
8. Evaluating ANN quality: recall@k vs. speed
9. Choosing an index for your use case
10. 🏆 Capstone: benchmark 3 index types on a larger corpus


In [ ]:
import numpy as np
import time
np.random.seed(0)
print("Ready.")

---
## Chapter 1 — Why Brute-Force Doesn't Scale

📖 **Theory.** Brute-force search computes the similarity between your query and **every single
vector** in the database — O(n) per query, where n is the number of vectors. At n=1,000 this
takes microseconds. At n=100,000,000 (real production scale), even a fast dot product per vector
adds up to seconds — too slow for a user waiting on a search box or an LLM waiting on RAG
retrieval.

🖼️ **Diagram — brute force cost growth**
```
 n=1,000        ▏  (fast)
 n=1,000,000    ▏▏▏▏▏▏▏▏▏▏  (noticeable)
 n=100,000,000  ▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏▏  (too slow for real-time)
                cost grows LINEARLY with database size
```

🧠 **Mental model.** Brute-force is like reading every book in a library to find one fact.
**Approximate Nearest Neighbor (ANN)** algorithms are like using the library's catalog system —
you don't check every book, just the ones the catalog points you toward, and you occasionally
miss the single best book in exchange for massive speed.


In [ ]:
def brute_force_search(query, vectors, k=5):
    sims = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query) + 1e-9)
    top_k = np.argsort(-sims)[:k]
    return top_k, sims[top_k]

# time brute force at increasing scale
for n in [1_000, 50_000, 200_000]:
    vectors = np.random.randn(n, 128).astype(np.float32)
    query = np.random.randn(128).astype(np.float32)
    start = time.time()
    idx, sims = brute_force_search(query, vectors, k=5)
    elapsed = time.time() - start
    print(f"n={n:>9,}  brute-force search time: {elapsed*1000:.2f} ms")

### ✏️ Your Turn 1.1
In a comment, explain why a search that takes 50ms per query is a problem for a RAG system that
needs to serve many concurrent users.

In [ ]:
# your explanation


✅ **Solution**
```python
# 50ms per query means only ~20 queries/second can be served by one process at full load.
# With many concurrent users (or a RAG pipeline making multiple retrieval calls per request),
# this quickly becomes a bottleneck -- latency stacks up and users wait.
```

---
## Chapter 2 — The Accuracy/Speed Tradeoff: Approximate Search

📖 **Theory.** ANN algorithms trade a small amount of **accuracy** (they might occasionally
miss the true single best match) for a huge amount of **speed**. This tradeoff is measured by
**recall@k**: what fraction of the *true* top-k nearest neighbors does the approximate method
actually return?

🖼️ **Diagram — the core tradeoff**
```
 exact (brute force):  100% recall,  slow    (checks everything)
 approximate (ANN):    ~95% recall,  fast    (checks a smart subset)
                        ↑ small accuracy loss, ↑ big speed gain
```

🧠 **Mental model.** Nearly every ANN algorithm's design question is: **"how do I avoid checking
every vector, while still checking the ones most likely to be the true nearest neighbors?"** The
different algorithms (next chapters) each answer this differently.


In [ ]:
def recall_at_k(approx_results, exact_results):
    approx_set, exact_set = set(approx_results), set(exact_results)
    return len(approx_set & exact_set) / len(exact_set)

# illustrative: an approximate method that "misses" one of the true top-5
exact_top5 = [42, 17, 8, 99, 3]
approx_top5 = [42, 17, 8, 50, 3]   # found 4 of 5 true neighbors, missed #99, found #50 instead
print("recall@5:", recall_at_k(approx_top5, exact_top5))   # 0.8 -- 80% of true neighbors found

### ✏️ Your Turn 2.1
Compute recall@5 for an approximate method that found `[42, 17, 8, 99, 3]` — identical to
`exact_top5`.

In [ ]:
perfect_recall = None
print(perfect_recall)

✅ **Solution**
```python
perfect_recall = recall_at_k([42,17,8,99,3], exact_top5)   # 1.0
```

---
## Chapter 3 — Clustering-Based Search: IVF (Inverted File Index)

📖 **Theory.** **IVF** pre-clusters all vectors (using k-Means, from the Classical ML lab!) into
`n_clusters` groups, each with a **centroid**. At search time: (1) find the closest few
centroids to your query, (2) only search vectors **within those clusters** — skipping everything
else entirely.

🖼️ **Diagram — IVF search**
```
 INDEX BUILD:  cluster all vectors into groups, remember each group's centroid
               [cluster A: •••]  [cluster B: •••]  [cluster C: •••]

 SEARCH:       query ─► find nearest centroid(s), e.g. cluster B
               ─► only compare against cluster B's vectors (skip A and C entirely)
```


In [ ]:
from sklearn.cluster import KMeans

class IVFIndex:
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters
    def build(self, vectors):
        self.vectors = vectors
        self.kmeans = KMeans(n_clusters=self.n_clusters, n_init=5, random_state=0)
        self.cluster_labels = self.kmeans.fit_predict(vectors)
        self.centroids = self.kmeans.cluster_centers_
    def search(self, query, k=5, n_probe=2):
        # 1. find the n_probe closest centroids to the query
        centroid_sims = self.centroids @ query
        probe_clusters = np.argsort(-centroid_sims)[:n_probe]
        # 2. only search vectors within those clusters
        candidate_idx = np.where(np.isin(self.cluster_labels, probe_clusters))[0]
        candidate_vectors = self.vectors[candidate_idx]
        sims = candidate_vectors @ query
        top_local = np.argsort(-sims)[:k]
        return candidate_idx[top_local]

np.random.seed(0)
vectors = np.random.randn(5000, 64).astype(np.float32)
query = np.random.randn(64).astype(np.float32)

ivf = IVFIndex(n_clusters=20)
ivf.build(vectors)
results = ivf.search(query, k=5, n_probe=3)
print("IVF top-5 results (indices):", results)
print(f"searched only {len(np.where(np.isin(ivf.cluster_labels, np.argsort(-(ivf.centroids @ query))[:3]))[0])} of {len(vectors)} vectors")

⚡ **Pro tip.** `n_probe` (how many clusters to search) is the main accuracy/speed knob:
`n_probe=1` is fastest but riskiest (the true nearest neighbor might be in a *different*
cluster near the boundary); higher `n_probe` recovers accuracy at the cost of speed.

⚠️ **Common trap.** If a query lands near a cluster **boundary**, its true nearest neighbor
might be in the *neighboring* cluster, not its own — this is exactly why `n_probe > 1` matters.

### ✏️ Your Turn 3.1
Build an `IVFIndex` with `n_clusters=50` and compare search results using `n_probe=1` vs
`n_probe=5` for the same query. Do they return the same top-5?

In [ ]:
ivf2 = IVFIndex(n_clusters=50)
ivf2.build(vectors)
results_probe1 = None
results_probe5 = None
print(results_probe1); print(results_probe5)

✅ **Solution**
```python
results_probe1 = ivf2.search(query, k=5, n_probe=1)
results_probe5 = ivf2.search(query, k=5, n_probe=5)
# they often differ -- probe=5 usually finds better/more results since it checks more territory
```

---
## Chapter 4 — Locality-Sensitive Hashing (LSH)

📖 **Theory.** **LSH** uses random hash functions specially designed so that **similar vectors
are likely to land in the same hash bucket** (unlike normal hashing, which scatters similar
items randomly). At search time, only compare against vectors sharing the query's hash bucket.

A simple LSH scheme for cosine similarity: pick random hyperplanes; a vector's hash bit is
"which side of the hyperplane it's on" (sign of the dot product).

🖼️ **Diagram — random hyperplane hashing**
```
        hyperplane
             │
    • A      │      • B     A and C are on the same side -> same hash bit (likely similar)
             │                B is on the other side -> different hash bit
    • C      │
```


In [ ]:
class SimpleLSH:
    def __init__(self, n_hyperplanes=8, dim=64, seed=0):
        rng = np.random.RandomState(seed)
        self.hyperplanes = rng.randn(n_hyperplanes, dim)   # random hyperplane normal vectors
    def hash_vector(self, v):
        # sign of dot product with each hyperplane -> a bit string
        bits = (self.hyperplanes @ v > 0).astype(int)
        return tuple(bits)
    def build(self, vectors):
        self.vectors = vectors
        self.buckets = {}
        for i, v in enumerate(vectors):
            h = self.hash_vector(v)
            self.buckets.setdefault(h, []).append(i)
    def search(self, query, k=5):
        h = self.hash_vector(query)
        candidate_idx = self.buckets.get(h, [])
        if not candidate_idx:
            return np.array([])   # empty bucket -- a known LSH limitation
        candidates = self.vectors[candidate_idx]
        sims = candidates @ query
        top_local = np.argsort(-sims)[:k]
        return np.array(candidate_idx)[top_local]

lsh = SimpleLSH(n_hyperplanes=6, dim=64)
lsh.build(vectors)
print("number of buckets used:", len(lsh.buckets))
print("avg vectors per bucket:", len(vectors) / len(lsh.buckets))
lsh_results = lsh.search(query, k=5)
print("LSH top-5 results:", lsh_results)

⚠️ **Common trap.** With too few hyperplanes, buckets are large (slow, like brute force on a
subset). With too many, buckets can be **empty** for a given query (as handled above) — you get
zero results even though good matches exist elsewhere. Real systems use **multiple hash tables**
in parallel to reduce this "empty bucket" risk.

### ✏️ Your Turn 4.1
Build an LSH index with `n_hyperplanes=10` (more hyperplanes = smaller, more precise buckets).
Compare the average bucket size to the `n_hyperplanes=6` version.

In [ ]:
lsh2 = SimpleLSH(n_hyperplanes=10, dim=64)
lsh2.build(vectors)
avg_bucket_size = None
print(avg_bucket_size)

✅ **Solution**
```python
lsh2 = SimpleLSH(n_hyperplanes=10, dim=64)
lsh2.build(vectors)
avg_bucket_size = len(vectors) / len(lsh2.buckets)   # smaller than the 6-hyperplane version
```

---
## Chapter 5 — Graph-Based Search: HNSW (the industry standard)

📖 **Theory.** **HNSW (Hierarchical Navigable Small World)** is what FAISS, Pinecone, Weaviate,
and Chroma use by default — it's the current state of the art for ANN search. The idea: build a
**multi-layer graph** where each vector is a node connected to its approximate neighbors. Search
starts at the **top layer** (few nodes, long-range connections — like an express highway) and
**greedily walks toward the query**, dropping down a layer each time it can't improve further —
like using highways to get close, then local streets to arrive precisely.

🖼️ **Diagram — the HNSW layered graph**
```
 Layer 2 (sparse, long jumps):    A ────────────► D              "highways"
                                  │                │
 Layer 1 (medium):          A ─► B ─► C ─────────► D ─► E        "main roads"
                             │    │    │            │    │
 Layer 0 (dense, all nodes): A─B─C─F─G─H─I─D─J─K─L─E─M─N          "local streets"

 search: start top layer, greedily move toward query, drop a layer when stuck, repeat
```

🧠 **Mental model.** It's how you'd navigate a city: take the highway to get to the right
neighborhood fast, then walk the local streets for the final precise steps — instead of walking
the entire city grid from the start.


In [ ]:
class SimpleHNSW:
    """A simplified approximation of HNSW's graph search: build a k-NN graph, then
    do a MULTI-ENTRY-POINT best-first (beam) search across it. Real HNSW uses multiple
    LAYERS (sparse long-range links on top, dense short-range links at the bottom) to
    get this same 'don't get stuck in one neighborhood' robustness even more efficiently
    -- our multiple random entry points approximate that same effect for teaching clarity."""
    def __init__(self, n_neighbors=16):
        self.n_neighbors = n_neighbors
    def build(self, vectors):
        self.vectors = vectors
        n = len(vectors)
        # build an approximate k-NN graph: each node connects to its n_neighbors closest vectors
        # (in real HNSW this is built incrementally and cleverly; we brute-force it here for clarity)
        self.graph = {}
        sims_all = vectors @ vectors.T
        for i in range(n):
            neighbors = np.argsort(-sims_all[i])[1:self.n_neighbors+1]   # skip self (index 0)
            self.graph[i] = list(neighbors)
    def search(self, query, k=5, n_entry_points=5, ef=40, seed=0):
        rng = np.random.RandomState(seed)
        entry_points = rng.choice(len(self.vectors), min(n_entry_points, len(self.vectors)), replace=False)
        visited = set(entry_points.tolist())
        frontier = [(float(self.vectors[e] @ query), int(e)) for e in entry_points]
        best = list(frontier)
        steps = 0
        while frontier and steps < ef:
            frontier.sort(key=lambda x: -x[0])
            sim, node = frontier.pop(0)
            steps += 1
            for nb in self.graph[node]:
                if nb in visited: continue
                visited.add(nb)
                nb_sim = float(self.vectors[nb] @ query)
                frontier.append((nb_sim, nb))
                best.append((nb_sim, nb))
        best.sort(key=lambda x: -x[0])
        return np.array([n for _, n in best[:k]])

hnsw = SimpleHNSW(n_neighbors=16)
hnsw.build(vectors[:2000])   # smaller subset for a quick illustrative build
hnsw_results = hnsw.search(query, k=5, n_entry_points=5, ef=40)
print("HNSW beam-search results:", hnsw_results)

⚡ **Pro tip.** HNSW's key advantage: **logarithmic** search time (roughly O(log n)) instead
of linear O(n) — this is *why* it scales to billions of vectors while brute force can't. The
tradeoff is more complex, memory-heavier index construction.

⚠️ **Common trap.** Greedy graph walks can get stuck in **local optima** — a node that looks
best among its immediate neighbors but isn't the true global best. Real HNSW mitigates this with
multiple layers and multiple entry points, improving the odds of escaping local optima.

### ✏️ Your Turn 5.1
Run the HNSW search with two **different** random seeds for entry-point selection (e.g. `seed=0` and
`seed=7`). Do they converge to similar top-5 results?

In [ ]:
results_seed0 = None
results_seed7 = None
print(results_seed0); print(results_seed7)

✅ **Solution**
```python
results_seed0 = hnsw.search(query, k=5, n_entry_points=5, ef=40, seed=0)
results_seed7 = hnsw.search(query, k=5, n_entry_points=5, ef=40, seed=7)
# usually similar/overlapping, but not always identical -- illustrates why real HNSW's
# multi-layer structure (rather than random restarts) gives more CONSISTENT convergence.
```

---
## Chapter 6 — Product Quantization (Compressing Vectors)

📖 **Theory.** Storing millions of full-precision (float32) vectors uses a lot of **memory**.
**Product Quantization (PQ)** compresses vectors: split each vector into `m` sub-vectors, run
k-Means on each sub-vector *space* separately to build a small codebook, then store each
sub-vector as just its **codebook index** (a few bits) instead of the full floats.

🖼️ **Diagram — splitting and quantizing**
```
 original vector (8 floats):  [0.2, -0.5, 0.8, 0.1, -0.3, 0.9, 0.4, -0.1]
                                └──sub1──┘ └──sub2──┘ └──sub3──┘ └──sub4──┘
 each sub-vector -> nearest codebook centroid -> store just the centroid's ID (e.g. 1 byte each)
 compressed:  [codeword_3, codeword_7, codeword_2, codeword_5]   <- tiny vs 8 floats!
```


In [ ]:
class ProductQuantizer:
    def __init__(self, n_subvectors=4, n_clusters=16):
        self.m = n_subvectors
        self.n_clusters = n_clusters
    def fit(self, vectors):
        dim = vectors.shape[1]
        assert dim % self.m == 0
        self.sub_dim = dim // self.m
        self.codebooks = []
        for i in range(self.m):
            sub = vectors[:, i*self.sub_dim:(i+1)*self.sub_dim]
            km = KMeans(n_clusters=self.n_clusters, n_init=3, random_state=0).fit(sub)
            self.codebooks.append(km)
    def encode(self, v):
        codes = []
        for i in range(self.m):
            sub = v[i*self.sub_dim:(i+1)*self.sub_dim]
            code = self.codebooks[i].predict(sub.reshape(1,-1))[0]
            codes.append(code)
        return codes
    def decode(self, codes):
        return np.concatenate([self.codebooks[i].cluster_centers_[codes[i]] for i in range(self.m)])

pq = ProductQuantizer(n_subvectors=4, n_clusters=16)
pq.fit(vectors[:1000])

original = vectors[42]
codes = pq.encode(original)
reconstructed = pq.decode(codes)

original_bytes = original.nbytes
compressed_bytes = len(codes) * 1   # 1 byte per code index (16 clusters fits in 4 bits, rounded to 1 byte)
print("original vector size:", original_bytes, "bytes")
print("compressed representation:", compressed_bytes, "bytes")
print(f"compression ratio: {original_bytes/compressed_bytes:.0f}x smaller")
print("reconstruction error (L2):", np.linalg.norm(original - reconstructed).round(4))

⚠️ **Common trap.** PQ is **lossy compression** — the reconstructed vector is an
approximation, not exact. There's a real accuracy cost in exchange for the memory savings, which
matters when a database can't fit in RAM otherwise.

### ✏️ Your Turn 6.1
Fit a `ProductQuantizer` with `n_subvectors=8` (finer split) instead of 4. Compare the
reconstruction error to the 4-subvector version — does finer splitting reduce error?

In [ ]:
pq2 = ProductQuantizer(n_subvectors=8, n_clusters=16)
pq2.fit(vectors[:1000])
codes2 = None
reconstructed2 = None
error2 = None
print(error2)

✅ **Solution**
```python
codes2 = pq2.encode(original)
reconstructed2 = pq2.decode(codes2)
error2 = np.linalg.norm(original - reconstructed2)
# typically LOWER error than the 4-subvector version -- finer splits capture more detail
```

---
## Chapter 7 — Combining Techniques: IVF + PQ

📖 **Theory.** Production vector databases **combine** techniques for best results: use **IVF**
to narrow the search to a few clusters (speed), then store vectors **within** those clusters as
**PQ codes** (memory savings), computing approximate distances directly on the compressed codes.
This is literally what FAISS's popular `IVFPQ` index does.

🖼️ **Diagram — IVF narrows, PQ compresses**
```
 IVF:  narrow search to ~2-3 relevant clusters out of thousands   (speed)
 PQ:   store each vector in those clusters as a tiny compressed code  (memory)
 combined: fast AND memory-efficient -- how billion-vector databases work
```


In [ ]:
class IVFPQIndex:
    def __init__(self, n_clusters=20, n_subvectors=4, pq_clusters=16):
        self.ivf = IVFIndex(n_clusters=n_clusters)
        self.pq = ProductQuantizer(n_subvectors=n_subvectors, n_clusters=pq_clusters)
    def build(self, vectors):
        self.ivf.build(vectors)
        self.pq.fit(vectors)
        self.codes = np.array([self.pq.encode(v) for v in vectors])
    def search(self, query, k=5, n_probe=3):
        # use IVF to narrow candidates, decode PQ codes only for those candidates
        centroid_sims = self.ivf.centroids @ query
        probe_clusters = np.argsort(-centroid_sims)[:n_probe]
        candidate_idx = np.where(np.isin(self.ivf.cluster_labels, probe_clusters))[0]
        decoded = np.array([self.pq.decode(self.codes[i]) for i in candidate_idx])
        sims = decoded @ query
        top_local = np.argsort(-sims)[:k]
        return candidate_idx[top_local]

ivfpq = IVFPQIndex(n_clusters=20, n_subvectors=4, pq_clusters=16)
ivfpq.build(vectors[:2000])
results = ivfpq.search(query, k=5, n_probe=3)
print("IVF+PQ results:", results)
print("memory per vector: full=", vectors.shape[1]*4, "bytes  vs  PQ code=", ivfpq.pq.m, "bytes")

### ✏️ Your Turn 7.1
In a comment, explain why combining IVF and PQ is better than using either alone for a
100-million-vector production database.

In [ ]:
# your explanation


✅ **Solution**
```python
# IVF alone is fast but still stores full-precision vectors -- expensive in RAM at scale.
# PQ alone saves memory but still must scan compressed codes across ALL vectors -- slow.
# Combined: IVF narrows to a few relevant clusters (speed), PQ shrinks what's stored
# within them (memory) -- together they solve both bottlenecks at billion-vector scale.
```

---
## Chapter 8 — Evaluating ANN Quality: recall@k vs. Speed

📖 **Theory.** No single "best" index exists — every choice trades recall for speed. The
standard way to evaluate: compute **exact** (brute-force) results as ground truth, then measure
**recall@k** and **query time** for each approximate method, and plot recall vs. speed.

🖼️ **Diagram — the recall/speed frontier**
```
 recall
   1.0 │ brute force (exact, slow)
       │      •
  0.95 │           • HNSW (great tradeoff)
       │                 • IVF
  0.80 │                       • LSH (fastest, lowest recall)
       └──────────────────────────────► speed (queries/sec)
```


In [ ]:
def benchmark_index(name, search_fn, query, exact_top5, n_runs=20):
    start = time.time()
    for _ in range(n_runs):
        results = search_fn(query)
    elapsed = (time.time() - start) / n_runs
    recall = recall_at_k(list(results), list(exact_top5))
    return {"index": name, "avg_query_time_ms": round(elapsed*1000, 3), "recall@5": round(recall, 2)}

test_vectors = vectors[:2000]
test_query = query

exact_idx, _ = brute_force_search(test_query, test_vectors, k=5)

ivf_bench = IVFIndex(n_clusters=20); ivf_bench.build(test_vectors)
lsh_bench = SimpleLSH(n_hyperplanes=8, dim=64); lsh_bench.build(test_vectors)

results_table = [
    benchmark_index("Brute Force", lambda q: brute_force_search(q, test_vectors, k=5)[0], test_query, exact_idx),
    benchmark_index("IVF", lambda q: ivf_bench.search(q, k=5, n_probe=3), test_query, exact_idx),
    benchmark_index("LSH", lambda q: lsh_bench.search(q, k=5), test_query, exact_idx),
]
for row in results_table:
    print(row)

### ✏️ Your Turn 8.1
Add `SimpleHNSW` to the benchmark table above (built on `test_vectors`), and print its recall@5
and query time alongside the others.

In [ ]:
hnsw_bench = SimpleHNSW(n_neighbors=10)
hnsw_bench.build(test_vectors)
hnsw_row = None
print(hnsw_row)

✅ **Solution**
```python
hnsw_bench = SimpleHNSW(n_neighbors=10)
hnsw_bench.build(test_vectors)
hnsw_row = benchmark_index("HNSW", lambda q: hnsw_bench.search(q, k=5, n_entry_points=5, ef=40), test_query, exact_idx)
print(hnsw_row)
```

---
## Chapter 9 — Choosing an Index for Your Use Case

📖 **Theory.** A practical decision framework:

| Situation | Recommended index |
|---|---|
| < 10K vectors, exact results needed | Brute force (simplest, no accuracy loss) |
| Medium scale, need good speed/accuracy balance | **HNSW** (default choice in most vector DBs today) |
| Huge scale, RAM-constrained | **IVF+PQ** (FAISS's `IndexIVFPQ`) |
| Very high-dimensional, binary-ish data | LSH variants |
| Frequently updated/streaming data | HNSW (supports incremental inserts well) |

🧠 **Mental model.** Start with brute force for correctness/prototyping. Move to **HNSW** as your
default at real scale — it's what Chroma, Pinecone, Weaviate, and FAISS's `IndexHNSWFlat` use by
default for good reason: strong recall, solid speed, and reasonably easy to maintain.


In [ ]:
def recommend_index(n_vectors, memory_constrained, needs_exact):
    if needs_exact or n_vectors < 10_000:
        return "Brute Force"
    if memory_constrained and n_vectors > 10_000_000:
        return "IVF + PQ"
    return "HNSW"

scenarios = [
    (5_000, False, True),
    (500_000, False, False),
    (50_000_000, True, False),
]
for n, mem, exact in scenarios:
    print(f"n={n:>12,}  memory_constrained={mem!s:5}  needs_exact={exact!s:5}  -> {recommend_index(n, mem, exact)}")

### ✏️ Your Turn 9.1
What would `recommend_index` return for `n_vectors=2_000_000, memory_constrained=False,
needs_exact=False`?

In [ ]:
answer = None
print(answer)

✅ **Solution**
```python
answer = recommend_index(2_000_000, False, False)   # "HNSW"
```

---
## 🏆 Chapter 10 — Capstone: Benchmark 3 Index Types on a Larger Corpus

Build a benchmark comparing **Brute Force**, **IVF**, and **HNSW** on a 5,000-vector corpus:
report each index's build time, average query time, and recall@5 versus the exact brute-force
answer. This is exactly the kind of evaluation real teams run before choosing a production
vector database index.

In [ ]:
np.random.seed(10)
# Use a lower, more realistic dimension and a query built by perturbing a real corpus point
# (like a real "find similar items" query) -- this mirrors how real embedding spaces behave
# much better than pure random noise in very high dimensions would.
corpus_vectors = np.random.randn(5000, 32).astype(np.float32)
corpus_vectors /= np.linalg.norm(corpus_vectors, axis=1, keepdims=True)
bench_query = corpus_vectors[123] + np.random.RandomState(1).randn(32).astype(np.float32) * 0.1
bench_query /= np.linalg.norm(bench_query)
print("corpus ready:", corpus_vectors.shape)

### ✏️ Capstone Tasks
1. Compute the exact top-5 (brute force) as ground truth.
2. Build an `IVFIndex` (`n_clusters=30`) and an `SimpleHNSW` (`n_neighbors=12`) on the corpus,
   timing each build.
3. Benchmark all three (brute force, IVF, HNSW): average query time over several runs + recall@5.
4. Print a comparison table and state which index you'd recommend for this corpus size.

In [ ]:
# Your benchmark pipeline here


✅ **Capstone Solution**
```python
# 1. exact ground truth
exact_top5, _ = brute_force_search(bench_query, corpus_vectors, k=5)

# 2. build indexes, timing each
t0 = time.time()
ivf_final = IVFIndex(n_clusters=30); ivf_final.build(corpus_vectors)
ivf_build_time = time.time() - t0

t0 = time.time()
hnsw_final = SimpleHNSW(n_neighbors=24); hnsw_final.build(corpus_vectors)
hnsw_build_time = time.time() - t0

# 3. benchmark query time + recall
final_results = [
    {"index": "Brute Force", "build_s": 0.0,
     **{k:v for k,v in benchmark_index("bf", lambda q: brute_force_search(q, corpus_vectors, k=5)[0], bench_query, exact_top5).items() if k != "index"}},
    {"index": "IVF", "build_s": round(ivf_build_time, 3),
     **{k:v for k,v in benchmark_index("ivf", lambda q: ivf_final.search(q, k=5, n_probe=4), bench_query, exact_top5).items() if k != "index"}},
    {"index": "HNSW", "build_s": round(hnsw_build_time, 3),
     **{k:v for k,v in benchmark_index("hnsw", lambda q: hnsw_final.search(q, k=5, n_entry_points=5, ef=40), bench_query, exact_top5).items() if k != "index"}},
]
print(f"{'Index':12s} {'Build(s)':>9s} {'Query(ms)':>10s} {'Recall@5':>9s}")
for row in final_results:
    print(f"{row['index']:12s} {row['build_s']:>9.3f} {row['avg_query_time_ms']:>10.3f} {row['recall@5']:>9.2f}")

print("\\nFor 5,000 vectors: brute force is still fast enough and exact -- at this scale,")
print("the added complexity of IVF/HNSW isn't yet justified. The crossover point is")
print("typically in the 100K-1M+ vector range, per Chapter 9's guidance.")
```

🎉 **You understand production vector search!** Why brute force doesn't scale, the accuracy/speed
tradeoff, IVF clustering, LSH hashing, HNSW graph search (the industry default), Product
Quantization compression, combined IVF+PQ, and how to evaluate and choose an index. This is
exactly what FAISS, Pinecone, Chroma, and Weaviate implement (at production-grade scale and
optimization) under their APIs.

---
### 📌 Concept Quick-Reference
**Why ANN:** brute force is O(n) per query — too slow at real scale
**Tradeoff:** recall@k (accuracy) vs. query speed — no free lunch
**IVF:** k-Means clusters vectors; search only the nearest few clusters (n_probe controls tradeoff)
**LSH:** random hyperplane hashing puts similar vectors in the same bucket
**HNSW:** multi-layer graph; greedy walk from sparse top layer to dense bottom layer (industry default)
**PQ:** split vectors into sub-vectors, quantize each via k-Means -> tiny compressed codes
**IVF+PQ:** combine narrowing (speed) with compression (memory) — how FAISS's IndexIVFPQ works
**Choosing:** brute force (small/exact) -> HNSW (general default) -> IVF+PQ (huge + memory-constrained)
